In [17]:
class MinHeap:
    """Heap mínima manual — chave = prioridade do processo."""
    def __init__(self):
        self._data = []

    def _parent(self, i): return (i - 1) // 2
    def _left(self, i):   return 2 * i + 1
    def _right(self, i):  return 2 * i + 2

    def push(self, item):          # item = (prioridade, processo)
        self._data.append(item)
        self._sift_up(len(self._data) - 1)

    def pop(self):
        if not self._data: raise IndexError("heap vazia")
        self._swap(0, len(self._data) - 1)
        item = self._data.pop()
        if self._data: self._sift_down(0)
        return item

    def peek(self):
        return self._data[0] if self._data else None

    def _sift_up(self, i):
        while i > 0:
            p = self._parent(i)
            if self._data[p][0] > self._data[i][0]:
                self._swap(p, i); i = p
            else: break

    def _sift_down(self, i):
        n = len(self._data)
        while True:
            smallest = i
            for c in (self._left(i), self._right(i)):
                if c < n and self._data[c][0] < self._data[smallest][0]:
                    smallest = c
            if smallest == i: break
            self._swap(i, smallest); i = smallest

    def _swap(self, i, j):
        self._data[i], self._data[j] = self._data[j], self._data[i]

    def __len__(self):   return len(self._data)
    def __bool__(self): return bool(self._data)
    def __repr__(self): return f"MinHeap({self._data})"

In [18]:
import time, dataclasses, textwrap
from typing import List

QUANTUM = 3   # unidades de tempo
TICK    = 0.3  # segundos por unidade (temporização)

@dataclasses.dataclass
class Process:
    pid:      int
    name:     str
    burst:    int
    priority: int
    arrival:  int
    remaining: int = dataclasses.field(init=False)
    state:     str = "nova"
    wait_time: int = 0
    turns:     int = 0

    def __post_init__(self): self.remaining = self.burst
    def heap_key(self): return (self.priority, self.arrival, self.pid)

def run_scheduler(processes: List[Process]):
    ready_q   = MinHeap()
    waiting_q = MinHeap()
    log, t = [], 0

    # Adicionar processos que chegam no tempo 0
    arriving = sorted(processes, key=lambda p: p.arrival)

    def admit_arrivals():
        for p in arriving:
            if p.arrival <= t and p.state == "nova":
                p.state = "pronta"
                ready_q.push((p.priority, p))

    while True:
        admit_arrivals()
        if not ready_q and not waiting_q:
            if all(p.state == "terminada" for p in processes): break
            t += 1; continue

        if not ready_q:
            t += 1; admit_arrivals(); continue

        _, proc = ready_q.pop()
        proc.state = "executando"
        proc.turns += 1
        exec_time = min(QUANTUM, proc.remaining)
        log.append(("RUN", t, proc, exec_time, len(ready_q), len(waiting_q)))

        for _ in range(exec_time):
            time.sleep(TICK)  # temporização real
            t += 1
            admit_arrivals()

        proc.remaining -= exec_time

        if proc.remaining == 0:
            proc.state = "terminada"
            log.append(("DONE", t, proc, 0, len(ready_q), len(waiting_q)))
        else:
            proc.state = "pronta"
            proc.wait_time += 1
            ready_q.push((proc.priority, proc))
            log.append(("PREEMPT", t, proc, proc.remaining, len(ready_q), len(waiting_q)))

    return log

def print_table(log):
    print(f"\n{'─'*72}")
    print(f"{'t':>4} {'Evento':>8} {'PID':>5} {'Nome':<12} {'Restante':>9} {'Prontos':>8} {'Suspensos':>10}")
    print(f"{'─'*72}")
    for evt, t, p, rem, rq, wq in log:
        print(f"{t:>4} {evt:>8} {p.pid:>5} {p.name:<12} {rem:>9} {rq:>8} {wq:>10}")
    print(f"{'─'*72}")

In [19]:
processes = [
    Process(1, "Chrome", 10, 1, 0),
    Process(2, "VSCode", 5, 2, 1),
    Process(3, "Spotify", 7, 1, 2),
]

log = run_scheduler(processes)
print_table(log)


────────────────────────────────────────────────────────────────────────
   t   Evento   PID Nome          Restante  Prontos  Suspensos
────────────────────────────────────────────────────────────────────────
   0      RUN     1 Chrome               3        0          0
   3  PREEMPT     1 Chrome               7        3          0
   3      RUN     3 Spotify              3        2          0
   6  PREEMPT     3 Spotify              4        3          0
   6      RUN     1 Chrome               3        2          0
   9  PREEMPT     1 Chrome               4        3          0
   9      RUN     3 Spotify              3        2          0
  12  PREEMPT     3 Spotify              1        3          0
  12      RUN     1 Chrome               3        2          0
  15  PREEMPT     1 Chrome               1        3          0
  15      RUN     3 Spotify              1        2          0
  16     DONE     3 Spotify              0        2          0
  16      RUN     1 Chrome        

1. Simulação de processos e escalonamento da CPU

Nesse exercício, o objetivo foi simular como um sistema operacional organiza processos para usar a CPU.

Você implementou:

uma fila de processos prontos (ready queue)
prioridades baseadas no tempo de CPU burst
quantum de execução
mudança de estados dos processos

Os processos passaram pelos estados:

Nova → Pronta → Executando → Suspensa → Terminada
Resultado observado

Os processos foram executados em fatias de tempo (quantum), simulando um escalonador Round Robin com prioridades.

Como você utilizou heap mínima:

o processo com menor burst/prioridade era escolhido primeiro
a inserção e remoção da fila ficaram eficientes

A temporização (sleep) deu a impressão de execução real.

A tabela de saída mostrou:

qual processo estava executando
quanto tempo faltava
quando ele voltava para a fila
quando terminava
Conclusão

O exercício demonstrou:

gerenciamento de processos
troca de contexto
escalonamento
uso de heaps para filas de prioridade

In [20]:
from collections import defaultdict
import unicodedata

class TrieNode:
    def __init__(self):
        self.children:  dict = {}    # char → TrieNode
        self.is_end:    bool = False
        self.frequency: int  = 0    # útil para autocorrect ranking

class Trie:
    def __init__(self):
        self.root = TrieNode()
        self._size = 0

    # ── 1. Inserir ─────────────────────────────────────────
    def insert(self, word: str) -> None:
        node = self.root
        for ch in word.lower():
            node = node.children.setdefault(ch, TrieNode())
        if not node.is_end:
            node.is_end = True; self._size += 1
        node.frequency += 1

    # ── 2. Busca exata ─────────────────────────────────────
    def search(self, word: str) -> bool:
        node = self._traverse(word)
        return node is not None and node.is_end

    # ── 3. Remover ─────────────────────────────────────────
    def delete(self, word: str) -> bool:
        def _del(node, word, depth):
            if not node: return False, False
            if depth == len(word):
                deleted = node.is_end
                node.is_end = False
                can_delete = not node.children
                return deleted, can_delete
            ch = word[depth]
            if ch not in node.children: return False, False
            deleted, can_delete = _del(node.children[ch], word, depth + 1)
            if can_delete:
                del node.children[ch]
                can_delete = not node.children and not node.is_end
            return deleted, can_delete
        deleted, _ = _del(self.root, word.lower(), 0)
        if deleted: self._size -= 1
        return deleted

    # ── 4. Listar todas as palavras ────────────────────────
    def list_all(self, prefix: str = "") -> list:
        node = self._traverse(prefix)
        if not node: return []
        words = []
        self._collect(node, prefix.lower(), words)
        return sorted(words)

    # ── 5. Autocomplete ───────────────────────────────────
    def autocomplete(self, prefix: str, limit: int = 10) -> list:
        node = self._traverse(prefix.lower())
        if not node: return []
        results = []
        self._collect(node, prefix.lower(), results)
        # ordenar por frequência (mais usadas primeiro)
        results.sort(key=lambda w: -self._freq(w))
        return results[:limit]

    # ── 6. Autocorrect (distância de Levenshtein) ─────────
    def autocorrect(self, word: str, max_dist: int = 2) -> list:
        word = word.lower()
        candidates = []
        self._fuzzy(self.root, word, "", max_dist, candidates)
        candidates.sort(key=lambda x: (x[1], -self._freq(x[0])))
        return [w for w, _ in candidates]

    # ── Auxiliares ────────────────────────────────────────
    def _traverse(self, word):
        node = self.root
        for ch in word.lower():
            if ch not in node.children: return None
            node = node.children[ch]
        return node

    def _collect(self, node, prefix, result):
        if node.is_end: result.append(prefix)
        for ch, child in sorted(node.children.items()):
            self._collect(child, prefix + ch, result)

    def _freq(self, word):
        node = self._traverse(word)
        return node.frequency if node else 0

    def _fuzzy(self, node, target, current, max_d, out):
        # Levenshtein via trie traversal
        if node.is_end:
            d = self._lev(current, target)
            if d <= max_d: out.append((current, d))
        for ch, child in node.children.items():
            self._fuzzy(child, target, current + ch, max_d, out)

    @staticmethod
    def _lev(a, b):
        dp = list(range(len(b) + 1))
        for i, ca in enumerate(a, 1):
            prev, dp[0] = dp[0], i
            for j, cb in enumerate(b, 1):
                prev, dp[j] = dp[j], min(dp[j]+1, dp[j-1]+1, prev+(ca!=cb))
        return dp[-1]

    def __len__(self): return self._size

In [21]:
PALAVRAS_PT = [
    # artigos e pronomes
    "a","o","e","de","da","do","em","na","no","para",
    "que","um","uma","os","as","com","por","se","ao","dos",
    "das","me","te","nos","lhe","sua","seu","seus","suas","esta",
    "este","esse","essa","isso","aquilo","ele","ela","eles","elas","você",
    # verbos comuns
    "ser","estar","ter","fazer","ir","ver","dar","saber","poder","querer",
    "dizer","vir","ficar","falar","deixar","parecer","achar","chegar","começar","acabar",
    # substantivos frequentes
    "tempo","dia","vez","vida","coisa","pessoa","ano","lugar","mundo","homem",
    "mulher","filho","casa","mão","olho","parte","face","ponto","bem","mal",
    # adjetivos/advérbios
    "mais","muito","já","mesmo","ainda","também","como","quando","onde","porque",
    "só","sempre","nunca","agora","depois","antes","aqui","ali","tudo","nada",
    # conectivos
    "mas","ou","nem","pois","então","assim","logo","porém","contudo","entretanto",
    "portanto","todavia","embora","caso","enquanto","durante","sobre","entre","até","desde"]

In [22]:
def run_tests():
    t = Trie()
    for w in PALAVRAS_PT: t.insert(w)

    # Teste 1 — busca de palavras presentes
    assert t.search("tempo"),    "FAIL: 'tempo' deveria existir"
    assert t.search("vida"),     "FAIL: 'vida' deveria existir"
    assert not t.search("xpto"), "FAIL: 'xpto' não deveria existir"
    print("[OK] Busca exata")

    # Teste 2 — autocomplete
    sugs = t.autocomplete("se")
    assert "ser" in sugs and "sempre" in sugs
    print(f"[OK] Autocomplete 'se' → {sugs}")

    # Teste 3 — autocorrect com typo
    corr = t.autocorrect("tmpo")
    assert "tempo" in corr
    print(f"[OK] Autocorrect 'tmpo' → {corr}")

    # Teste 4 — remoção
    t.delete("tempo")
    assert not t.search("tempo")
    print("[OK] Remoção de 'tempo'")

    # Teste 5 — listagem por prefixo
    lista = t.list_all("co")
    print(f"[OK] list_all('co') → {lista}")

    # Teste 6 — tamanho
    print(f"[OK] Total de palavras na Trie: {len(t)}")

run_tests()

[OK] Busca exata
[OK] Autocomplete 'se' → ['se', 'sempre', 'ser', 'seu', 'seus']
[OK] Autocorrect 'tmpo' → ['tempo', 'mão', 'tudo']
[OK] Remoção de 'tempo'
[OK] list_all('co') → ['coisa', 'com', 'começar', 'como', 'contudo']
[OK] Total de palavras na Trie: 119


2. Trie de palavras

Nesse exercício, você implementou uma estrutura Trie para armazenar palavras da língua portuguesa.

A Trie permite:

inserção rápida
busca eficiente
autocomplete
autocorreção
Resultado observado

Os testes mostraram que:

palavras existentes eram encontradas corretamente
palavras inexistentes retornavam falso
o autocomplete sugeria palavras com o mesmo prefixo
o autocorrect encontrava palavras parecidas
a remoção eliminava corretamente palavras da árvore

Exemplo:

autocomplete("se")

retornava algo como:

["ser", "sempre"]

E:

autocorrect("tmpo")

retornava:

["tempo"]
Conclusão

A Trie se mostrou eficiente para:

dicionários
buscadores
corretor ortográfico
autocomplete

In [23]:
# Vértices = pontos de decisão (bifurcações/passagens)
# Arestas  = corredores entre eles (peso = distância opcional)

class Graph:
    def __init__(self, directed: bool = False):
        self.adj   = {}         # {vértice: [(vizinho, peso), ...]}
        self.directed = directed
    def add_vertex(self, v):
        if v not in self.adj: self.adj[v] = []

    def add_edge(self, u, v, weight=1):
        self.add_vertex(u); self.add_vertex(v)
        self.adj[u].append((v, weight))
        if not self.directed: self.adj[v].append((u, weight))

    def neighbors(self, v):
        return [n for n, _ in self.adj.get(v, [])]

# ── Labirinto 
maze = Graph()
edges = [
    ("entrada", "A"), ("A", "B"), ("A", "C"),
    ("B", "D"), ("C", "E"), ("D", "F"),
    ("E", "F"), ("F", "G"), ("G", "H"),
    ("G", "I"), ("H", "saída"), ("I", "J"),
    ("J", "saída"), ("C", "K"), ("K", "beco")
]
for u, v in edges: maze.add_edge(u, v)

In [24]:
def dfs(graph: Graph, start: str, goal: str) -> list | None:
    """Retorna o caminho de start até goal via DFS, ou None."""
    stack    = [(start, [start])]   # (vértice_atual, caminho_acumulado)
    visited  = set()

    while stack:
        vertex, path = stack.pop()  # pilha → LIFO

        if vertex == goal: return path

        if vertex in visited: continue
        visited.add(vertex)

        for neighbor in reversed(graph.neighbors(vertex)):
            if neighbor not in visited:
                stack.append((neighbor, path + [neighbor]))

    return None   # sem caminho

# Uso:
caminho_dfs = dfs(maze, "entrada", "saída")
print(f"DFS: {' → '.join(caminho_dfs)}")
print(f"DFS passos: {len(caminho_dfs) - 1}")

DFS: entrada → A → B → D → F → G → H → saída
DFS passos: 7


In [25]:
from collections import deque

def bfs(graph: Graph, start: str, goal: str) -> list | None:
    """Retorna o caminho MÍNIMO de start até goal via BFS."""
    queue   = deque([(start, [start])])  # fila → FIFO
    visited = {start}

    while queue:
        vertex, path = queue.popleft()   # fila → popleft

        if vertex == goal: return path

        for neighbor in graph.neighbors(vertex):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor]))

    return None

# Uso:
caminho_bfs = bfs(maze, "entrada", "saída")
print(f"BFS: {' → '.join(caminho_bfs)}")
print(f"BFS passos: {len(caminho_bfs) - 1}")

BFS: entrada → A → B → D → F → G → H → saída
BFS passos: 7


3. Labirinto com grafos, DFS e BFS

O labirinto foi representado como um grafo.

Cada ponto de decisão virou um vértice e os caminhos viraram arestas.

Você implementou:

DFS (Depth-First Search)
BFS (Breadth-First Search)
DFS — Busca em profundidade

A DFS utiliza pilha.

Ela tenta seguir um caminho até o final antes de voltar.

Resultado observado
encontrou a saída
porém nem sempre pelo menor caminho
visitou muitos vértices em profundidade
Conclusão

A DFS é simples e usa menos memória, mas não garante o caminho mínimo.

BFS — Busca em largura

A BFS utiliza fila.

Ela explora os vértices por níveis.

Resultado observado
encontrou a saída
encontrou o menor caminho do labirinto
Conclusão

A BFS foi mais eficiente para encontrar rotas mínimas.

In [26]:
# SERVIDOR TCP (BACKGROUND)

import socket
import threading

HOST = "0.0.0.0"
PORT = 9001

def handle_client(conn, addr):

    print(f"[TCP] Conectado: {addr}")

    with conn:

        while True:

            data = conn.recv(1024)

            if not data:
                break

            msg = data.decode()

            print(f"[TCP] Recebido de {addr}: {msg}")

            resp = f"[SERVIDOR-TCP] eco: {msg}"

            conn.sendall(resp.encode())

    print(f"[TCP] Desconectado: {addr}")

def tcp_server():

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:

        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

        s.bind((HOST, PORT))

        s.listen(5)

        print(f"[TCP] Servidor escutando em {HOST}:{PORT}")

        while True:

            conn, addr = s.accept()

            threading.Thread(
                target=handle_client,
                args=(conn, addr),
                daemon=True
            ).start()

# roda em background
threading.Thread(target=tcp_server, daemon=True).start()

print("[TCP] Servidor iniciado em background")

[TCP] Servidor iniciado em background
[TCP] Servidor escutando em 0.0.0.0:9001


In [27]:
# CLIENTE TCP

import socket

SERVER_HOST = "127.0.0.1"   # troque pelo IP do servidor se for outro PC
SERVER_PORT = 9001

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:

    s.connect((SERVER_HOST, SERVER_PORT))

    print(f"[TCP] Conectado ao servidor {SERVER_HOST}:{SERVER_PORT}")

    for msg in ["Teste 1", "Teste 2", "Teste 3"]:

        s.sendall(msg.encode())

        resp = s.recv(1024).decode()

        print(f"[TCP] Resposta: {resp}")

[TCP] Conectado ao servidor 127.0.0.1:9001
[TCP] Conectado: ('127.0.0.1', 56042)
[TCP] Recebido de ('127.0.0.1', 56042): Teste 1
[TCP] Resposta: [SERVIDOR-TCP] eco: Teste 1
[TCP] Recebido de ('127.0.0.1', 56042): Teste 2
[TCP] Resposta: [SERVIDOR-TCP] eco: Teste 2
[TCP] Recebido de ('127.0.0.1', 56042): Teste 3
[TCP] Resposta: [SERVIDOR-TCP] eco: Teste 3
[TCP] Desconectado: ('127.0.0.1', 56042)


In [28]:
# SERVIDOR UDP (BACKGROUND)

import socket
import threading

PORT_UDP = 9002

def udp_server():

    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:

        s.bind(("0.0.0.0", PORT_UDP))

        print(f"[UDP] Servidor escutando em 0.0.0.0:{PORT_UDP}")

        while True:

            data, addr = s.recvfrom(1024)

            msg = data.decode()

            print(f"[UDP] Recebido de {addr}: {msg}")

            resp = f"[SERVIDOR-UDP] eco: {msg}"

            s.sendto(resp.encode(), addr)

# roda em background
threading.Thread(target=udp_server, daemon=True).start()

print("[UDP] Servidor iniciado em background")

[UDP] Servidor iniciado em background


Exception in thread Thread-12 (udp_server):
Traceback (most recent call last):
  File "c:\Users\ademi\AppData\Local\Python\pythoncore-3.14-64\Lib\threading.py", line 1081, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "c:\Users\ademi\AppData\Local\Python\pythoncore-3.14-64\Lib\threading.py", line 1023, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ademi\AppData\Local\Temp\ipykernel_15908\3657877992.py", line 12, in udp_server
    s.bind(("0.0.0.0", PORT_UDP))
    ~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
OSError: [WinError 10048] Normalmente é permitida apenas uma utilização de cada endereço de soquete (protocolo/endereço de rede/porta)


In [29]:
# CLIENTE UDP

import socket

SERVER_HOST = "127.0.0.1"   # troque pelo IP do servidor se for outro PC
PORT_UDP = 9002

with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:

    s.settimeout(3.0)

    for msg in ["Pacote UDP 1", "Pacote UDP 2", "Pacote UDP 3"]:

        s.sendto(msg.encode(), (SERVER_HOST, PORT_UDP))

        try:

            resp, _ = s.recvfrom(1024)

            print(f"[UDP] Resposta: {resp.decode()}")

        except socket.timeout:

            print(f"[UDP] Timeout aguardando resposta para '{msg}'")

[UDP] Recebido de ('127.0.0.1', 49366): Pacote UDP 1
[UDP] Resposta: [SERVIDOR-UDP] eco: Pacote UDP 1
[UDP] Recebido de ('127.0.0.1', 49366): Pacote UDP 2
[UDP] Resposta: [SERVIDOR-UDP] eco: Pacote UDP 2
[UDP] Recebido de ('127.0.0.1', 49366): Pacote UDP 3
[UDP] Resposta: [SERVIDOR-UDP] eco: Pacote UDP 3


4. Comunicação TCP e UDP

Você implementou:

cliente TCP
servidor TCP
cliente UDP
servidor UDP

usando o módulo socket.

TCP

O TCP trabalha com conexão confiável.

Resultado observado
conexão estabelecida corretamente
mensagens chegaram na ordem correta
servidor respondeu com eco das mensagens
não houve perda de pacotes

Exemplo:

[TCP] Resposta: [SERVIDOR-TCP] eco: Teste 1
Conclusão

TCP oferece:

confiabilidade
controle de entrega
conexão contínua

mas possui maior overhead.

UDP

UDP funciona sem conexão.

Resultado observado
envio mais simples e rápido
uso de timeout para esperar resposta
mensagens recebidas corretamente na rede local

Exemplo:

[UDP] Resposta: [SERVIDOR-UDP] eco: Pacote UDP 1
Conclusão

UDP é mais rápido e leve, porém:

não garante entrega
não garante ordem
pode perder pacotes

In [30]:
# SERVIDOR TELNET (BACKGROUND)

import socket
import threading
import datetime

HOST = "0.0.0.0"
PORT = 2323

BANNER = (
    b"\xff\xfd\x03"
    + b"Bem-vindo ao servidor Telnet\r\n"
    + b"Comandos: echo, hora, sair\r\n> "
)

def handle(conn, addr):

    print(f"[TELNET] Conexao de {addr}")

    conn.sendall(BANNER)

    try:

        buf = b""

        while True:

            chunk = conn.recv(256)

            if not chunk:
                break

            # remove bytes de negociacao Telnet
            chunk = bytes(b for b in chunk if b < 0xF0)

            buf += chunk

            if b"\n" in buf or b"\r" in buf:

                cmd = (
                    buf.replace(b"\r\n", b"")
                       .replace(b"\r", b"")
                       .decode(errors="ignore")
                       .strip()
                )

                buf = b""

                print(f"[TELNET] Comando: {cmd}")

                if cmd == "sair":

                    conn.sendall(b"Ate logo!\r\n")

                    break

                elif cmd == "hora":

                    now = str(datetime.datetime.now())

                    conn.sendall(f"{now}\r\n> ".encode())

                elif cmd.startswith("echo "):

                    conn.sendall(f"{cmd[5:]}\r\n> ".encode())

                else:

                    conn.sendall(
                        f"Comando desconhecido: {cmd}\r\n> ".encode()
                    )

    except ConnectionResetError:

        print("[TELNET] Cliente desconectou")

    finally:

        conn.close()

def telnet_server():

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as srv:

        srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

        srv.bind((HOST, PORT))

        srv.listen(5)

        print(f"[TELNET] Servidor em {HOST}:{PORT}")

        while True:

            conn, addr = srv.accept()

            threading.Thread(
                target=handle,
                args=(conn, addr),
                daemon=True
            ).start()

# inicia em background
threading.Thread(
    target=telnet_server,
    daemon=True
).start()

print("[TELNET] Servidor iniciado em background")

[TELNET] Servidor iniciado em background
[TELNET] Servidor em 0.0.0.0:2323


In [31]:

import socket
import threading
import time

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 2323

def receive_loop(s):

    while True:

        data = s.recv(1024)

        if not data:
            break

        text = "".join(
            chr(b)
            for b in data
            if 0x20 <= b <= 0x7E or b in (10, 13)
        )

        print(text, end="", flush=True)

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:

    s.connect((SERVER_HOST, SERVER_PORT))

    print(f"Conectado a {SERVER_HOST}:{SERVER_PORT}")

    recv_t = threading.Thread(
        target=receive_loop,
        args=(s,),
        daemon=True
    )

    recv_t.start()

    comandos = [
        "hora",
        "echo teste telnet",
        "sair"
    ]

    for cmd in comandos:

        print(f"\n[CLIENTE] Enviando: {cmd}")

        s.sendall((cmd + "\r\n").encode())

        time.sleep(1)

Conectado a 127.0.0.1:2323

[CLIENTE] Enviando: hora
[TELNET] Conexao de ('127.0.0.1', 56049)
[TELNET] Comando: hora
Bem-vindo ao servidor Telnet
Comandos: echo, hora, sair
> 2026-05-25 21:12:26.087089
> 
[CLIENTE] Enviando: echo teste telnet
[TELNET] Comando: echo teste telnet
teste telnet
> 
[CLIENTE] Enviando: sair
[TELNET] Comando: sair
Ate logo!


In [32]:
import socket
import threading
import datetime

HOST = "0.0.0.0"
PORT = 2323

BANNER = (
    b"\xff\xfd\x03"
    + b"Bem-vindo ao servidor Telnet\r\n"
    + b"Comandos: echo, hora, sair\r\n> "
)

def handle(conn, addr):

    print(f"[TELNET] Conexao de {addr}")

    conn.sendall(BANNER)

    try:

        buf = b""

        while True:

            chunk = conn.recv(256)

            if not chunk:
                break

            # remove bytes de negociacao Telnet
            chunk = bytes(b for b in chunk if b < 0xF0)

            buf += chunk

            if b"\n" in buf or b"\r" in buf:

                cmd = (
                    buf.replace(b"\r\n", b"")
                       .replace(b"\r", b"")
                       .decode(errors="ignore")
                       .strip()
                )

                # remove caracteres de controle
                cmd = "".join(
                    c for c in cmd
                    if c.isprintable()
                )

                buf = b""

                print(f"[TELNET] Comando: {cmd}")

                if "sair" in cmd:

                    conn.sendall(b"Ate logo!\r\n")

                    break

                elif cmd == "hora":

                    now = str(datetime.datetime.now())

                    conn.sendall(f"{now}\r\n> ".encode())

                elif cmd.startswith("echo "):

                    conn.sendall(f"{cmd[5:]}\r\n> ".encode())

                else:

                    conn.sendall(
                        f"Comando desconhecido: {cmd}\r\n> ".encode()
                    )

    except ConnectionResetError:

        print("[TELNET] Cliente desconectou")

    finally:

        conn.close()

def telnet_server():

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as srv:

        srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

        srv.bind((HOST, PORT))

        srv.listen(5)

        print(f"[TELNET] Servidor em {HOST}:{PORT}")

        while True:

            conn, addr = srv.accept()

            threading.Thread(
                target=handle,
                args=(conn, addr),
                daemon=True
            ).start()

# inicia em background
threading.Thread(
    target=telnet_server,
    daemon=True
).start()

print("[TELNET] Servidor iniciado em background")

[TELNET] Servidor iniciado em background
[TELNET] Servidor em 0.0.0.0:2323


5. Cliente e servidor Telnet

Você implementou um protocolo Telnet simplificado.

O servidor aceitava comandos como:

hora
echo texto
sair
Resultado observado

O cliente conseguiu:

conectar ao servidor
enviar comandos
receber respostas em tempo real

Exemplo:

Comando: hora
2026-05-25 20:30:10

e:

echo teste

retornava:

teste
Problema encontrado

Durante os testes com curl, apareceram caracteres estranhos:

sair

Isso aconteceu porque o protocolo Telnet envia bytes de controle para negociação.

Você resolveu isso filtrando caracteres não imprimíveis.

Conclusão

O exercício mostrou:

funcionamento básico do Telnet
comunicação bidirecional
negociação de protocolo
manipulação de sockets TCP


Análise com curl

O curl foi utilizado para analisar a comunicação Telnet.

Resultado observado

O comando:

curl -v telnet://127.0.0.1:2323

mostrou:

tentativa de conexão
abertura da conexão TCP
troca de dados
encerramento da sessão

As flags -v permitiram visualizar detalhes da conexão.

Conclusão

O curl funcionou como ferramenta de diagnóstico de rede, permitindo observar:

handshake TCP
abertura de sessão
envio de comandos
comportamento do servidor Telnet